In [1]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
# from src.aco_optimization import ACO
# Import Internal Modules
from src.preprocessing import DataPreprocessor
from src.embedding import PCAEmbedder, UMAPEmbedder
from src.optimization import PSO
from src.clustering import DBSCAN
from src.evaluation import (
    silhouette_score as sk_silhouette,
    davies_bouldin_index,
    calinski_harabasz_index
)
from src.visualization import (
    plot_clusters_3d,
    plot_full_report,
    plot_cluster_profiles  # <--- IMPORT THE NEW FUNCTION
)
from src.utils import Logger, Timer
from config import CONFIG

# 1. Setup Logging & Timer
logger = Logger(verbose=True)
timer = Timer()

try:
    # ---------------------------------------------------------
    # STEP 1: Load & Preprocess Data
    # ---------------------------------------------------------
    logger.info("1. Loading & Preprocessing Data...")
    preprocessor = DataPreprocessor()
    
    # 1. Load Data
    raw_df = preprocessor.load_data('data/Cleaned_pasien_large.csv') 
    
    # --- PERBAIKAN PENTING ---
    # Kita harus membuang NaN di sini agar 'df' sinkron dengan hasil transform.
    # Jika preprocessor Anda melakukan dropna(), kita harus melakukannya juga di sini.
    df = raw_df.dropna().reset_index(drop=True)

    
    logger.info(f"   Original Rows: {len(raw_df)} -> Clean Rows: {len(df)}")

    # 2. Transform data yang SUDAH bersih
    X_scaled = preprocessor.transform(df)
    
    # ---------------------------------------------------------
    # STEP 2: Embedding (Switchable: PCA or UMAP)
    # ---------------------------------------------------------
    method = CONFIG['embedding'].get('method', 'pca')
    logger.info(f"2. Applying {method.upper()} Embedding...")

    if method == 'umap':
        umap_cfg = CONFIG['embedding']['umap']
        embedder = UMAPEmbedder(
            n_neighbors=umap_cfg.get('n_neighbors', 15),
            min_dist=umap_cfg.get('min_dist', 0.1),
            n_components=umap_cfg.get('n_components', 5)
        )
    else:
        # Default to PCA
        pca_cfg = CONFIG['embedding']['pca']
        embedder = PCAEmbedder(n_components=pca_cfg.get('n_components', 10))

    X_embedded = embedder.fit_transform(X_scaled)
    
    logger.info(f"   Data Shape after {method.upper()}: {X_embedded.shape}")
    
    from src.visualization import plot_objective_landscape
    fig_landscape = plot_objective_landscape(
        X=X_embedded,  # Gunakan hasil UMAP/PCA
        bounds=CONFIG['pso']['bounds'],
        objective_kwargs=CONFIG['objective'],
        resolution=5, 
        logger=logger
    )

# Tampilkan dan Simpan
    fig_landscape.show()

    # ---------------------------------------------------------
    # STEP 3: Hyperparameter Optimization (PSO)
    # ---------------------------------------------------------
    logger.info("3. Starting PSO Optimization...")
    timer.start()
    
    pso = PSO(
        **CONFIG['pso'],
        logger=logger,
        objective_kwargs=CONFIG['objective']
    )
    
    # aco_optimizer = ACO(**CONFIG['pso'], logger=logger, objective_kwargs=CONFIG['objective'])
    
    best_params = pso.optimize(X_embedded)
    # best_params = aco_optimizer.optimize(X_embedded)
    
    elapsed = timer.stop()
    logger.success(f"PSO Finished in {elapsed:.2f}s")
    logger.info(f"   Best Parameters: {best_params}")

    # ---------------------------------------------------------
    # STEP 4: Final Clustering
    # ---------------------------------------------------------
    logger.info("4. Running Final DBSCAN Clustering...")
    dbscan = DBSCAN()
    labels = dbscan.fit(
        X_embedded,
        eps=best_params['eps'],
        min_samples=best_params['min_samples']
    )
    
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    logger.success(f"   Clusters found: {n_clusters}")
    logger.info(f"   Noise points: {n_noise}")

    # ---------------------------------------------------------
    # STEP 5: Evaluation Metrics
    # ---------------------------------------------------------
    logger.info("5. Calculating Metrics...")
    
    if n_clusters > 1:
        mask = labels != -1
        metrics = {
            'silhouette': sk_silhouette(X_embedded[mask], labels[mask]) if mask.sum() > 2 else 0,
            'dbi': davies_bouldin_index(X_embedded, labels),
            'ch': calinski_harabasz_index(X_embedded, labels)
        }
    else:
        metrics = {'silhouette': 0, 'dbi': 0, 'ch': 0}
        
    logger.success(f"   Metrics: {metrics}")

    # ---------------------------------------------------------
    # STEP 6: Visualization & Reporting
    # ---------------------------------------------------------
    logger.info("6. Generating Visualizations...")
    
    # A. Interactive 3D Plot (Structure)
    logger.info("   Generating 3D Structure Plot...")
    pca_3d = PCA(n_components=3, random_state=42)
    X_plot_3d = pca_3d.fit_transform(X_scaled) 
    
    plot_clusters_3d(
        X_plot_3d, 
        labels, 
        title=f"3D PCA Clustering (Eps: {best_params['eps']:.2f})", 
        save_path="output/3d_clusters.html"
    )

    # B. Interactive Profile Plot (Radar & Boxplot) <-- NEW ADDITION
    logger.info("   Generating Cluster Profiles (Radar & Box)...")
    plot_cluster_profiles(
        df=df,              # Dataframe original (sudah bersih) untuk nilai metrik
        labels=labels,      # Hasil clustering dari DBSCAN

        show_plot=True      # Tampilkan plot interaktif juga
        
    )
    # C. Full Static Report (Matplotlib)
    logger.info("   Generating Technical Report...")
    plot_full_report(
        X_embedded=X_embedded, 
        labels=labels, 
        pso_history=pso.history, 
        metrics=metrics, 
        save_path= "output/full_report.png"
    )

    logger.success("=== ALL PROCESSES COMPLETED SUCCESSFULLY ===")

except Exception as e:
    logger.error(f"Pipeline Failed: {str(e)}")
    raise e

[00:29:00] INFO: 1. Loading & Preprocessing Data...
✓ Data loaded: 10637 rows, 100 columns
[00:29:00] INFO:    Original Rows: 10637 -> Clean Rows: 10637
PREPROCESSING (DataFrame Input)
✓ Numeric features selected: 100 columns
  Features: ['BADGE', 'KEHAMILAN_5', 'OLAHRAGA', 'ALERGI', 'TINGGI', 'BERAT', 'TENSI_1', 'TENSI_2', 'TENSI_3', 'TENSI_4', 'TENSI_5', 'NADI', 'PERNAPASAN', 'SUHU', 'KULIT_RAMBUT_2', 'PENYAKIT_MATA_2', 'CONJUNGTIVA', 'SCLERA_2', 'TELINGA_3', 'TELINGA_4', 'MEMBRAN_TYMPANI_4', 'REFLEK_CAHAYA_2', 'REFLEK_CAHAYA_4', 'SERUMEN_PLUG_0', 'SERUMEN_PLUG_5', 'HIDUNG_2', 'HIDUNG_3', 'SEPTUM_DEVIASI_4', 'CONCHA_3', 'CONCHA_4', 'CONCHA_5', 'POLYP_3', 'KERONGKONGAN_2', 'KERONGKONGAN_3', 'TONSIL_5', 'FARING_3', 'FARING_4', 'HERNIA_1', 'HAEMORROID', 'EPIDIDYMIS_TESTIS_PROSTAT_0', 'EPIDIDYMIS_TESTIS_PROSTAT_1', 'JVP_1', 'STRUMA_2', 'MULUT_3', 'GUSI_2', 'GUSI_3', 'BATAS_JANTUNG_1', 'SUARA_JANTUNG_MURMUR_3', 'AUSKULTASI_3', 'THORAX_PHOTO_4', 'THORAX_PHOTO_5', 'THORAX_PHOTO_6', 'SEMBAB_

[00:30:12] INFO: 3. Starting PSO Optimization...
[00:30:12] INFO: PSO started: particles=30, iter=14


PSO Optimization:   7%|▋         | 1/14 [01:26<18:40, 86.19s/it]

[00:31:38] INFO: Iter 1/14 - best_fitness=-0.9961


KeyboardInterrupt: 

In [5]:
X_scaled

array([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
        -1.83799987e+31,  0.00000000e+00, -0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00, -1.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00, -0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00, -1.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00, -0.00000000e+00],
       ...,
       [ 1.11022302e-16,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00, -0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00, -0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00, -0.00000000e+00]],
      shape=(10637, 100))

In [4]:
pca_3d = PCA(n_components=3, random_state=42)
X_plot_3d = pca_3d.fit_transform(X_scaled) 


dbscan = DBSCAN()
labels = dbscan.fit(
        X_embedded,
        eps=5,
        min_samples=40
    )
plot_clusters_3d(
        X_plot_3d, 
        labels, 
        title=f"3D PCA Clustering (Eps: 5.00, MinPts: 40)", 

    )

In [6]:
df

,BADGE,KEHAMILAN_5,OLAHRAGA,ALERGI,TINGGI,BERAT,TENSI_1,TENSI_2,TENSI_3,TENSI_4,...,GULA_DARAH_PUASA,GULA_DARAH_2JAMPP,URINE_REDUKSI_PUASA_3,URINE_REDUKSI_2JAMPP_4,ANY_EAR_ABNORMAL,ANY_NOSE_ABNORMAL,ANY_ORAL_ABNORMAL,ANY_THROAT_ABNORMAL,ANY_URINE_SEDIMENT_POS,ANY_GLUCOSE_URINE_POS
0,133191500.0,1,1,0.0,1660.0,800.0,0,0,0,0,...,860.0,870.0,1,1,0,0,0,0,1,0
1,133313090.0,1,0,0.0,1600.0,570.0,0,0,0,0,...,930.0,1050.0,1,1,0,0,0,1,1,0
2,133313090.0,1,0,0.0,1550.0,520.0,0,0,0,1,...,880.0,930.0,1,1,0,0,0,1,1,0
3,166996720.0,1,1,0.0,1590.0,570.0,0,0,0,1,...,1250.0,1970.0,1,1,0,0,0,1,1,0
4,166281850.0,1,1,0.0,1710.0,620.0,0,0,1,0,...,1070.0,1020.0,1,1,0,0,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10632,132901160.0,1,1,0.0,1650.0,730.0,0,0,0,1,...,980.0,970.0,1,1,0,0,0,1,1,0
10633,132901160.0,1,1,0.0,1570.0,580.0,0,0,0,1,...,920.0,870.0,1,1,0,0,0,1,1,0
10634,133401860.0,1,1,0.0,1720.0,820.0,0,0,0,1,...,970.0,950.0,1,1,0,0,0,1,1,0
10635,132699230.0,1,1,0.0,1790.0,860.0,0,1,0,1,...,1060.0,1080.0,1,1,0,0,0,1,1,0
